# Get To Know A Dataset: WashU Repeat Browser

![WashU Repeat Browser logo](assets/repeat-browser-logo.png)

A short introduction to discovering and loading the WashU Repeat Browser dataset from the public `washu-repeat-browser` bucket in Amazon S3 (`us-east-2`). The Registry entry uses the slug [`washu-repeat-browser`](https://registry.opendata.aws/washu-repeat-browser/).

[Repeat Browser](https://repeatbrowser.org/) · [Documentation](https://rb-doc.readthedocs.io/en/latest/) · [Publication](https://doi.org/10.1101/gr.279764.124)

### Q: How have you organized your dataset?

The bucket is organized first by reference assembly (`hg38` or `mm10`) and then by assay. ChIP-seq is divided into histone and transcription-factor collections.

```text
s3://washu-repeat-browser/
├── hg38/{atac-seq,cage-seq,newcage-seq,dnase-seq,chip-seq/{histone,tf}}/
├── mm10/{atac-seq,dnase-seq,chip-seq/{histone,tf}}/
├── documentation/
└── precomputedEnrichment/
```

A processed directory is a data entry, often linked to an ENCODE file accession; it is not necessarily an independent biological experiment.

In [ ]:
# Dependencies: s3fs>=2024.6.1, pandas>=2.0, matplotlib>=3.7, requests>=2.31
import os
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import s3fs
from IPython.display import display

BUCKET = os.environ.get("WASHU_REPEAT_BROWSER_BUCKET", "washu-repeat-browser")

fs = s3fs.S3FileSystem(anon=True)

In [ ]:
def child_names(prefix):
    return [item.rstrip("/").split("/")[-1] for item in fs.ls(prefix, detail=False)]

print("Top level:", child_names(BUCKET))
print("hg38:", child_names(f"{BUCKET}/hg38"))
print("mm10:", child_names(f"{BUCKET}/mm10"))

### Q: What data formats are present?

- **CSV**: assay metadata and provenance tables.
- **BigWig**: indexed genomic signal tracks for range-based access.
- **Zarr v2**: chunked TE-subfamily, consensus and locus-level values used by Repeat Browser.
- **JSON and indexed resources**: saved sessions and precomputed enrichment data.

`UNI` represents uniquely mapped reads; `ALL` includes unique and multi-mapped reads. Keep the representation consistent when comparing datasets.

### Q: Can you show an example of downloading and loading data?

This example reads the human histone ChIP-seq metadata table directly from S3 without downloading the entire collection.

In [ ]:
metadata_key = f"{BUCKET}/hg38/chip-seq/histone/hg38_Histone_Chipseq_all.csv"
with fs.open(metadata_key, "rb") as handle:
    metadata = pd.read_csv(handle)
print(f"Rows: {len(metadata):,}")
print("Columns:", metadata.columns.tolist())
display(metadata.head())

#### Zarr arrays and locus-level chunks

Zarr metadata describe each array's shape, chunks, type and compression. File-mode stores use `loci_<subfamily>`; ChIP-seq Experiment-mode stores use `signal_loci_<subfamily>` and `control_loci_<subfamily>`. Each locus array is flattened as repeating `chromosome, start, end, RPKM` values.

The following example anonymously reads and decodes a specific `signal_loci_*` chunk from the public AWS bucket.

In [ ]:
zarr_array = (
    "https://washu-repeat-browser.s3.us-east-2.amazonaws.com/"
    "hg38/chip-seq/histone/"
    "Processed_ENCFF032RWB_signal/ENCFF032RWB_signal.zarr/"
    "signal_loci_MER70B"
)
metadata_response = requests.get(f"{zarr_array}/.zarray", timeout=60)
metadata_response.raise_for_status()
chunk_response = requests.get(f"{zarr_array}/0", timeout=60)
chunk_response.raise_for_status()
array_metadata = metadata_response.json()
compressed = chunk_response.content
values = np.frombuffer(
    zlib.decompress(compressed), dtype=array_metadata["dtype"]
)[: array_metadata["shape"][0]]
loci = pd.DataFrame(
    values.reshape(-1, 4), columns=["chromosome", "start", "end", "RPKM"]
)
display(loci.head())
print(f"Decoded {len(loci):,} MER70B loci records.")

### Q: A picture is worth a thousand words. Show an informative visual.

The chart below summarizes the current 8,774 indexed entries by assembly and assay.

In [ ]:
counts = pd.DataFrame([
    ("hg38", "ATAC-seq", 443), ("hg38", "CAGE-seq", 2),
    ("hg38", "New CAGE-seq", 213), ("hg38", "DNase-seq", 2529),
    ("hg38", "Histone ChIP-seq", 70), ("hg38", "TF ChIP-seq", 2423),
    ("mm10", "ATAC-seq", 176), ("mm10", "DNase-seq", 1773),
    ("mm10", "Histone ChIP-seq", 960), ("mm10", "TF ChIP-seq", 185),
], columns=["assembly", "assay", "processed_entries"])

chart = counts.pivot(index="assay", columns="assembly", values="processed_entries").fillna(0)
ax = chart.plot(kind="barh", figsize=(9, 5), color=["#BB3E55", "#F3A58C"])
ax.set(title="WashU Repeat Browser data entries", xlabel="Indexed entries", ylabel="")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

### Q: What is one question that you have answered using these data?

**Question:** Can TE-centered views recover known regulatory relationships at both subfamily and locus levels?

In [ ]:
paper_examples = pd.DataFrame([
    {
        "example": "STAT1 binding at MER41B",
        "data": "STAT1 ChIP-seq in IFNG-stimulated K562, HeLa-S3, and CD14+ macrophages",
        "views": "Heatmap → Consensus → loci",
        "observation": "Figure 2: STAT1 enrichment over MER41B in HeLa cells",
    },
    {
        "example": "LTR12C activation",
        "data": "CAGE-seq: DMSO, DAC, SB939, and DAC+SB",
        "views": "Heatmap → chromosome → loci",
        "observation": "Figure 3: DAC+SB and DAC increase LTR12C signal",
    },
])
for _, row in paper_examples.iterrows():
    print(f"Example: {row['example']}")
    print(f"  Data: {row['data']}")
    print(f"  Views: {row['views']}")
    print(f"  Observation: {row['observation']}")
    print()

The publication used these views to recover two known biological patterns: STAT1 binding over MER41B after IFNG stimulation, and activation of LTR12C after epigenetic treatment. The first connects ChIP-seq summary and `signal_loci_*` data; the second connects CAGE-seq summary and ordinary `loci_*` data.

These are examples of exploratory visualization rather than standalone causal tests. See the [publication](https://genome.cshlp.org/content/early/2025/02/21/gr.279764.124) for the full analyses.

### Q: What is one unanswered question that could be addressed with these data?

**Community challenge:** Which TE subfamilies show biosample-specific regulatory activity, and is that activity supported at both the subfamily and locus levels?

A reproducible analysis should compare compatible assays and controls, keep `UNI` and `ALL` separate, validate across replicates and preserve source accessions.

## Citation

Shen J, Cheng S, Purushotham D, Zhuo X, Du AY, Zhang W, Li D, Wang T. *Exploring the epigenome profiles of repetitive elements with the WashU Repeat Browser*. Genome Research. 2025. https://doi.org/10.1101/gr.279764.124

The examples above use anonymous, read-only access to the public AWS Open Data bucket.